# CV Screening Using TF‑IDF & Cosine Similarity

This notebook demonstrates a simple **NLP-based CV screening system**.

Pipeline:
1. Import libraries
2. Extract text from CV PDFs
3. Clean the text
4. Prepare documents
5. TF‑IDF vectorization
6. Cosine similarity
7. Rank candidates


In [5]:
# These libraries are required for PDF text extraction, text preprocessing,
# data handling, and similarity calculation.


%pip install pdfplumber pandas scikit-learn


import pdfplumber
import re
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Function: Extract Text From PDF

In [1]:
# This function reads a PDF file and extracts all text from each page.
# The extracted text will later be used for NLP preprocessing.

def extract_text(pdf_path):
    text = ""

    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            content = page.extract_text()
            if content:
                text += content + " "

    return text


## 3. Function: Clean Text

In [6]:
# Text cleaning is necessary before vectorization.
# Steps performed:

# 1. Convert text to lowercase
# 2. Remove special characters and numbers
# 3. Remove extra spaces

def clean_text(text):

    text = text.lower()

    text = re.sub(r'\n', ' ', text)

    text = re.sub(r'[^a-zA-Z\s]', '', text)

    text = re.sub(r'\s+', ' ', text)

    return text.strip()


## 4. Load and Clean CV PDFs

In [8]:
# Here we extract raw text from each CV PDF and apply text cleaning.
# The cleaned text will be used for TF‑IDF vectorization.

import os

folder = r"C:\Users\Md Al Mamun\OneDrive\Desktop\EXAM IVS 8-3-26\CV_folder"

cv1 = extract_text(os.path.join(folder,"exam_cv_1.pdf"))
cv2 = extract_text(os.path.join(folder,"Rafi_Mehebub_Bhuiyan_CV_2.pdf"))
cv3 = extract_text(os.path.join(folder,"Sm_Nuruzzaman_Nobel_CV_3.pdf"))

cv1_clean = clean_text(cv1)
cv2_clean = clean_text(cv2)
cv3_clean = clean_text(cv3)


## 5. Job Description

In [128]:
# The job description is cleaned using the same preprocessing pipeline
# so that comparison between CVs and the job description becomes fair.

job_description = """
Python experience, numpy, pandas, SQL, TDD,
LLM systems, embeddings, vector databases,
machine learning, ranking algorithms,
kubernetes, java, python, generative AI
"""

job_clean = clean_text(job_description)


 6. Prepare Documents for Vectorization

In [133]:
# We combine all CV texts and the job description into a single list.
# This allows TF‑IDF to learn vocabulary across all documents.

documents = [cv1_clean, cv2_clean, cv3_clean, job_clean]


7. TF‑IDF Vectorization

In [130]:
# TF‑IDF converts text documents into numerical vectors.
# Important words receive higher weights than common words.

vectorizer = TfidfVectorizer(stop_words="english")

tfidf_matrix = vectorizer.fit_transform(documents)


## 8. Cosine Similarity Calculation

In [134]:
# Cosine similarity measures how similar two vectors are.
# We compare each CV vector with the job description vector.

job_vector = tfidf_matrix[-1]

cv_vectors = tfidf_matrix[:-1]

similarity = cosine_similarity(cv_vectors, job_vector)


## 9. Rank Candidates

In [135]:
# Similarity scores are used to rank candidates.
# Higher score means the CV matches the job description better.

scores = similarity.flatten()

results = pd.DataFrame({
    "Candidate": [
        "Md Shakil Ahmed (CV1)",
        "Rafi Mehebub Bhuiyan (CV2)",
        "SM Nuruzzaman Nobel (CV3)"
    ],
    "Similarity Score": scores
})

ranking = results.sort_values(by="Similarity Score", ascending=False)

print("===== CV Screening Result =====")
print(ranking)

print("\nTop Candidate:")
print(ranking.iloc[0])


===== CV Screening Result =====
                    Candidate  Similarity Score
0       Md Shakil Ahmed (CV1)          0.162303
2   SM Nuruzzaman Nobel (CV3)          0.013489
1  Rafi Mehebub Bhuiyan (CV2)          0.009010

Top Candidate:
Candidate           Md Shakil Ahmed (CV1)
Similarity Score                 0.162303
Name: 0, dtype: object
